In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 加载model,tokenizer
custom_model_name = "fine-tuned-models/nllb-200-distilled-600M/zh2ko_0818"

model = AutoModelForSeq2SeqLM.from_pretrained(custom_model_name)
tokenizer = AutoTokenizer.from_pretrained(custom_model_name)

In [2]:
from transformers import pipeline

translator = pipeline("translation", model=model, tokenizer=tokenizer,
                      src_lang=tokenizer.src_lang,
                      tgt_lang=tokenizer.tgt_lang,
                      device="cuda:0", max_length=400)

In [3]:
import pandas as pd

df = pd.read_csv("data/output/all_files_merged_zh-CN_ko.csv")

source = df["zh-CN"].to_list()  # 待翻译的句子
references = df["ko"].to_list()  # 标准的翻译

In [ ]:
from tqdm.notebook import tqdm

# 进行翻译并提取结果
translated = [translator(item)[0] for item in tqdm(source)]
translated_text = [item['translation_text'] for item in translated]

  0%|          | 0/77509 [00:00<?, ?it/s]

D:\LongtuKoreaTranslationModel\venv\lib\site-packages\transformers\pipelines\base.py:1101: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
Your input_length: 437 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)
Your input_length: 419 is bigger than 0.9 * max_length: 400. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


In [ ]:
df_compare = pd.DataFrame({
    "source": source,
    "references": references,
    "candidates": translated_text,
})
df_compare

In [ ]:
file_name = "translation_result_of_{0}".format(custom_model_name.replace("/", "_"))
df_compare.to_excel("{0}.xlsx".format(file_name), index=False)
df_compare.to_csv("{0}.csv".format(file_name), index=False)